In [1]:
import torch

# Check for NVIDIA GPU (CUDA)
print("CUDA Available:", torch.cuda.is_available())

# Check for Apple Silicon Mac GPU (MPS)
print("MPS Available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CUDA Available: True
MPS Available: False
Using device: cuda


In [2]:
import pandas as pd
from Bio import SeqIO
from modules.sequences import (
    split_train_val_test,
    remove_unknown_amino_acids
)
from modules.model import (
    init_transition_mat,
    init_emission_mat
)
from modules.qmodel import (
    build_hqmm,
    convert_to_torch,
    compress_phmm_kraus
)
from pathlib import Path
from modules.qevaluation import description_accuracy_hqmm
from modules.qtrain import train_hqmm

In [5]:
family_df = pd.read_csv('protein_families.csv')
    
model_dir = Path("saved_models/phqmm/main")
model_dir.mkdir(parents=True, exist_ok=True)

# Fixed optimization parameters
tau = 0.1
alpha = 0.8
max_iter = 15
tol = 1e-3
batch_size = 16

for i in range(len(family_df)):
    name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]
    
    # read the fasta files
    recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]
    
    # separate to train, validation, and test set
    train_seqs, val_seqs, test_seqs = split_train_val_test(recs)
    remove_train, train_seqs = remove_unknown_amino_acids(train_seqs)
    remove_val, val_seqs = remove_unknown_amino_acids(val_seqs)
    remove_test, test_seqs = remove_unknown_amino_acids(test_seqs)
    print(f'Total of removed sequences in {name}-{accession}')
    print(f'Train set: {remove_train}')
    print(f'Validation set: {remove_val}')
    print(f'Test set: {remove_test}')

    # ==========================================
    # 1. MODEL CONFIGURATION
    # ==========================================
    
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    
    n = match_len * 2 + 1
    s = 21  # 20 amino acids + $
    W = 2 # fix the size of Kraus operators per symbol
    
    # Initialize PHMM transition and emission matrices
    trans_matdf = init_transition_mat(match_len)
    emit_matdf = init_emission_mat(match_len, seed=2)
    
    symbols = list(emit_matdf.index)
    sym_idx = {
        symbol: i
        for i, symbol in enumerate(symbols)
    }
    
    BEGIN = sym_idx["^"]
    END = sym_idx["$"]
    
    assert BEGIN == 0
    assert sorted(
        idx for symbol, idx in sym_idx.items()
        if symbol != "^"
    ) == list(range(1, s + 1))
    
    # ==========================================
    # 2. ORIGINAL PHMM-DERIVED KRAUS OPERATORS
    # ==========================================
    
    # Construct the original W=n Kraus operators
    # using the square-root observable-operator method.
    
    K_phmm = build_hqmm(
        trans_matdf,
        emit_matdf
    )
    
    K_phmm = convert_to_torch(K_phmm)
    
    # Exclude the start symbol (^)
    K_phmm = {
        key: value
        for key, value in K_phmm.items()
        if key[1] != BEGIN
    }
    
    print(
        f"Original PHMM-derived HQMM: "
        f"n={n}, W={n}, operators={len(K_phmm)}"
    )
    
    results = {}
    histories = {}
    trained_models = {}
    
    # ==========================================
    # 4. INITIAL DENSITY MATRIX
    # ==========================================
    
    # Use the same initial state as the original
    # PHMM-derived HQMM.
    
    p = torch.zeros(
        (n, n),
        dtype=torch.complex64,
        device=device
    )
    
    p[0, 0] = 1.0

    K_initial = compress_phmm_kraus(
        K_phmm=K_phmm,
        n=n,
        W=W,
        device=device,
        seed=100 + W
    )

    K_initial = {
        key: value.detach().clone().to(
            device=device,
            dtype=torch.complex64
        )
        for key, value in K_initial.items()
    }

    # --------------------------------------
    # Verify Kraus completeness
    # --------------------------------------

    with torch.no_grad():
        S = torch.zeros(
            (n, n),
            dtype=p.dtype,
            device=device
        )

        for A in K_initial.values():
            S += A.conj().T @ A

        I = torch.eye(
            n,
            dtype=p.dtype,
            device=device
        )

        completeness_error = (
            torch.linalg.matrix_norm(S - I).item()
        )

        print(
            f"Completeness error: "
            f"{completeness_error:.6e}",
            flush=True
        )

        if not torch.allclose(
            S, I, atol=1e-4, rtol=1e-4
        ):
            raise ValueError(
                f"Kraus completeness failed for W={W}"
            )

    # --------------------------------------
    # Evaluate untrained model
    # --------------------------------------

    with torch.no_grad():
        da0 = description_accuracy_hqmm(
            p,
            K_initial,
            W,
            sym_idx,
            test_seqs
        )

    label = f'{name}-{accession}'

    results[f"{label} (untrained)"] = (
        float(da0.mean()),
        float(da0.std())
    )

    print(
        f"Initial Test DA: "
        f"{results[f'{label} (untrained)'][0]:.4f} "
        f"± {results[f'{label} (untrained)'][1]:.4f}",
        flush=True
    )

    # --------------------------------------
    # Train HQMM using COSM
    # --------------------------------------

    K_trained, hist = train_hqmm(
        p=p,
        K=K_initial,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=test_seqs,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    histories[label] = hist
    trained_models[label] = K_trained

    # --------------------------------------
    # Save checkpoint and history
    # --------------------------------------

    prefix = f"{name}_{accession}_m{match_len}_n{n}"

    model_path = model_dir / (
        f"{prefix}_hqmm.pt"
    )

    history_path = model_dir / (
        f"{prefix}_history.csv"
    )

    checkpoint = {
        "K": {
            key: value.detach().cpu().clone()
            for key, value in K_trained.items()
        },
        "p": p.detach().cpu().clone(),
        "n": n,
        "s": s,
        "W": W,
        "initialization": "phmm_compressed",
        "tau": tau,
        "alpha": alpha,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "tol": tol,
        "epochs_completed": len(hist) - 1,
    }

    torch.save(checkpoint, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)
    print(f"History saved: {history_path}", flush=True)

    # --------------------------------------
    # Evaluate trained model
    # --------------------------------------

    K_ref = next(iter(K_trained.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    K_eval = {
        key: value.to(
            device=K_ref.device,
            dtype=K_ref.dtype
        )
        for key, value in K_trained.items()
    }

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval,
            K_eval,
            W,
            sym_idx,
            test_seqs
        )

    results[label] = (
        float(da.mean()),
        float(da.std())
    )

    print(
        f"Trained Test DA: "
        f"{results[label][0]:.4f} "
        f"± {results[label][1]:.4f}",
        flush=True
    )

Total of removed sequences in DUF7563-PF24444
Train set: 0
Validation set: 0
Test set: 0
Original PHMM-derived HQMM: n=79, W=79, operators=1659
Completeness error: 9.815134e-07
Initial Test DA: 0.0049 ± 0.0042
Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14606.13  train_da_mean 0.0048  val_da_mean 0.0069
  iter   1  batch   1/5  loss 198.1097  tau 0.100000
  iter   1  batch   2/5  loss 167.5814  tau 0.100000
  iter   1  batch   3/5  loss 182.9916  tau 0.100000
  iter   1  batch   4/5  loss 194.2332  tau 0.100000
  iter   1  batch   5/5  loss 220.1457  tau 0.100000
iter   1  train     5.3s  log-lik -11159.32  train_da_mean 0.2451  val_da_mean 0.2264
  iter   2  batch   1/5  loss 140.4018  tau 0.080000
  iter   2  batch   2/5  loss 180.1099  tau 0.080000
  iter   2  batch   3/5  loss 135.0735  tau 0.080000
  iter   2  batch   4/5  loss 155.8317  tau 0.080000
  iter   2  batch   5/5  loss 159.4018  tau 0.080000
iter   2  train    12.0s  log-lik -10002.05  tr

KeyboardInterrupt: 